# Dual-GPU Parallel CBM + GFlowNet + Rule Regularization (Kaggle T4 x2)

Notebook mỏng (launcher) tối ưu cho cấu hình **Kaggle GPU T4 x2 (15GB + 15GB VRAM) và RAM 30GB**:
- **GPU 0 (15GB):** Huấn luyện độc lập **Seed A (42)** từ Stage 1 đến Stage 5.
- **GPU 1 (15GB):** Huấn luyện song song **Seed B (44)** từ Stage 1 đến Stage 5.
- **Cơ chế chống OOM:** Cô lập thiết bị bằng `CUDA_VISIBLE_DEVICES`, giới hạn `num_workers=2` mỗi tiến trình (tổng 4 workers, RAM < 6GB / 30GB), `batch_size=64` (~2.5GB / 15GB VRAM).
- **Sanity Check 1 iteration:** Chạy kiểm thử luồng dữ liệu trước khi train chính thức.

In [ ]:
# =========================================================================
# THAM SỐ THỰC NGHIỆM DUAL-GPU (Tự động sinh bởi generator script)
# =========================================================================
DATA_DIR = '/kaggle/input/datasets/dangduykhoab2207532/mykhmer/khmer'
CONCEPT_ANNOTATIONS_PATH = '/kaggle/input/datasets/dangduykhoab2207532/mykhmer/image_concepts.json'
NUM_CLASSES = 9
N_CONCEPTS = 61
BACKBONE = 'alexnet'

SEED_A = 42                     # Chạy trên GPU 0 (15GB VRAM)
SEED_B = 44                     # Chạy trên GPU 1 (15GB VRAM)

BATCH_SIZE = 64                 # Chiếm ~2.5GB VRAM (an toàn cho 15GB VRAM)
NUM_WORKERS = 2                 # 2 workers mỗi GPU (tổng 4 workers < 6GB RAM / 30GB)
NUM_EPOCHS = 50
PATIENCE = 5
CONCEPT_WEIGHT = 0.1
GIT_REPO = 'https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git'
GIT_BRANCH = 'main'


In [ ]:
# Kiểm tra dữ liệu ảnh và file nhãn concept đầu vào
from pathlib import Path

data_path = Path(DATA_DIR)
concept_path = Path(CONCEPT_ANNOTATIONS_PATH)

if not data_path.exists():
    possible_data = list(Path("/kaggle/input").glob("**/khmer"))
    if possible_data:
        DATA_DIR = str(possible_data[0])
        print(f"Tự động phát hiện DATA_DIR tại: {DATA_DIR}")
    else:
        print(f"CẢNH BÁO: Không tìm thấy DATA_DIR: {DATA_DIR}")
else:
    print(f"DATA_DIR hợp lệ tại: {DATA_DIR}")

if not concept_path.exists():
    possible_json = list(Path("/kaggle/input").glob("**/*concept*.json"))
    if possible_json:
        CONCEPT_ANNOTATIONS_PATH = str(possible_json[0])
        print(f"Tự động phát hiện file concept tại: {CONCEPT_ANNOTATIONS_PATH}")
    else:
        print(f"CẢNH BÁO: Không tìm thấy file concept: {CONCEPT_ANNOTATIONS_PATH}")
else:
    print(f"File concept hợp lệ tại: {CONCEPT_ANNOTATIONS_PATH}")

In [ ]:
# Clone repo & setup môi trường
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/kaggle/working/neuro_symbolic_mlops_l2_app")
if not PROJECT.exists():
    print("Đang clone repo mới nhất...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(PROJECT)], check=True)
print("Repo sẵn sàng tại:", PROJECT)

In [ ]:
# Cài đặt thư viện phụ thuộc & Kiểm tra phần cứng Dual GPU + RAM
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
import subprocess, sys
import importlib.metadata as md

# Ghi lại phiên bản numpy gốc của môi trường để không bị đổi
numpy_ver = md.version("numpy")
print("numpy gốc:", numpy_ver)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torchgfn", "tensordict", "dvclive", "dvc", "openpyxl", "psutil"],
    check=True,
)

import importlib
importlib.invalidate_caches()
for module_name in tuple(sys.modules):
    if module_name == "gfn" or module_name.startswith("gfn."):
        del sys.modules[module_name]

import torch
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Hệ thống CPU RAM: {ram_gb:.1f} GB (Ngưỡng an toàn tối đa: 30GB)")

n_gpus = torch.cuda.device_count()
print(f"Số lượng GPU: {n_gpus}")
assert n_gpus >= 2, "HÃY CHỌN ACCELERATOR: GPU T4 x2 trên bảng điều khiển bên phải của Kaggle!"

for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    vram = props.total_memory / (1024**3)
    print(f"  [GPU {i}] {props.name} | VRAM: {vram:.1f} GB")

In [ ]:
# Cập nhật base params.yaml với các thông số chống OOM
import yaml

with open("params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

params["data_dir"] = DATA_DIR
params["num_classes"] = NUM_CLASSES
params["num_epochs"] = NUM_EPOCHS
params["patience"] = PATIENCE
params["batch_size"] = BATCH_SIZE
params["num_workers"] = NUM_WORKERS  # Giới hạn 2 workers chống RAM OOM
params["baseline_comparison"]["selected_architecture"] = BACKBONE

# CBM & Concept configurations
params.setdefault("cbm", {})
params["cbm"]["n_concepts"] = N_CONCEPTS
params["cbm"]["concept_weight"] = CONCEPT_WEIGHT
params["cbm"]["concept_annotations_path"] = CONCEPT_ANNOTATIONS_PATH
params.setdefault("features", {})
params["features"]["source"] = "cbm"
params.setdefault("rule_penalty", {})
params["rule_penalty"]["target_model"] = "cbm"

with open("params.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(params, f, sort_keys=False, allow_unicode=True)

print("Đã cấu hình params.yaml: batch_size=64, num_workers=2 để chạy song song 2 seeds không OOM!")

In [ ]:
# Chạy smoke test trong tiến trình Python riêng để tránh lỗi import trong kernel notebook.
subprocess.run(
    [sys.executable, "-m", "scripts.cbm_dual_gpu_smoke_test",
     "--data-dir", DATA_DIR,
     "--concept-annotations-path", CONCEPT_ANNOTATIONS_PATH,
     "--backbone", BACKBONE,
     "--num-classes", str(NUM_CLASSES),
     "--batch-size", str(BATCH_SIZE),
     "--num-workers", str(NUM_WORKERS),
     "--seed-a", str(SEED_A),
     "--seed-b", str(SEED_B)],
    check=True,
)

In [ ]:
# =========================================================================
# CHẠY SONG SONG 2 SEEDS TRÊN 2 GPU ĐỘC LẬP (KHÔNG GÂY LỖI OOM)
# =========================================================================
!python -m pipelines.run_dual_gpu_cbm_experiment --config params.yaml --seeds {SEED_A} {SEED_B}

In [ ]:
# Chạy phần tổng hợp báo cáo trong tiến trình Python riêng.
subprocess.run(
    [sys.executable, "-m", "scripts.cbm_dual_gpu_report",
     "--seeds", str(SEED_A), str(SEED_B)],
    check=True,
)


In [ ]:
# In Top luật Concept từ tiến trình Python riêng.
subprocess.run(
    [sys.executable, "-m", "scripts.cbm_dual_gpu_report",
     "--seeds", str(SEED_A), str(SEED_B),
     "--top-rules-seed", str(SEED_A)],
    check=True,
)

In [ ]:
# Đóng gói toàn bộ artifacts của cả 2 seeds thành file zip để tải về
!zip -r -q /kaggle/working/dual_gpu_cbm_outputs.zip outputs_seed_*
print("\nĐÃ NÉN TOÀN BỘ KẾT QUẢ 2 SEEDS TẠI: /kaggle/working/dual_gpu_cbm_outputs.zip")
!ls -lh /kaggle/working/dual_gpu_cbm_outputs.zip